# RAG Example: Loading Your Own Data into ChromaDB

This notebook shows how to load **any text data** into ChromaDB and query it.

We'll use the famous "Wear Sunscreen" speech (Chicago Tribune, 1997) as our example dataset.

Original source: [Salem Public Library — 1999 Yearbook](http://history.salem.lib.oh.us/yearbooks/1999/2facesplits/1999op_Part9.pdf)

## Step 1 — Read and chunk the text

The text file has the speech split into paragraphs (separated by blank lines).
We split on `\n\n` so each paragraph becomes one chunk in our database.

In [2]:
with open("../data/city_database.txt") as f:
    text = f.read()

chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print(f"{len(chunks)} chunks loaded\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:80]}...\n")

50 chunks loaded

Chunk 0: CITY: Cape Town
COUNTRY: South Africa
REGION: Africa
POPULATION: 4337000
BEST_TI...

Chunk 1: CITY: Nairobi
COUNTRY: Kenya
REGION: Africa
POPULATION: 4397000
BEST_TIME_TO_VIS...

Chunk 2: CITY: Cairo
COUNTRY: Egypt
REGION: Africa
POPULATION: 10230000
BEST_TIME_TO_VISI...

Chunk 3: CITY: Marrakech
COUNTRY: Morocco
REGION: Africa
POPULATION: 928000
BEST_TIME_TO_...

Chunk 4: CITY: Lagos
COUNTRY: Nigeria
REGION: Africa
POPULATION: 15000000
BEST_TIME_TO_VI...

Chunk 5: CITY: Tunis
COUNTRY: Tunisia
REGION: Africa
POPULATION: 638000
BEST_TIME_TO_VISI...

Chunk 6: CITY: Addis Ababa
COUNTRY: Ethiopia
REGION: Africa
POPULATION: 5200000
BEST_TIME...

Chunk 7: CITY: Johannesburg
COUNTRY: South Africa
REGION: Africa
POPULATION: 9570000
BEST...

Chunk 8: CITY: Accra
COUNTRY: Ghana
REGION: Africa
POPULATION: 2400000
BEST_TIME_TO_VISIT...

Chunk 9: CITY: Dakar
COUNTRY: Senegal
REGION: Africa
POPULATION: 1200000
BEST_TIME_TO_VIS...

Chunk 10: CITY: Paris
COUNTRY: France
REGI

## Step 2 — Load into ChromaDB

In [3]:
import chromadb

client = chromadb.PersistentClient("../chroma")

# Delete collection if it already exists (so we can re-run this cell)
try:
    client.delete_collection("city_db")
except Exception:
    pass

collection = client.create_collection(name="city_db")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Added {collection.count()} chunks to the 'city_db' collection")

2026-03-09 13:44:25.970046844 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Added 50 chunks to the 'city_db' collection


## Step 3 — Query the collection

ChromaDB uses the query text to find the most semantically similar chunks.
This is the same pattern we use in `5_rag.ipynb` with the nutrition database.

In [5]:
results = collection.query(
    query_texts=["What does the database say about Paris?"],
    n_results=1,
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Result {i + 1}:")
    print(doc)
    print()

Result 1:
CITY: Paris
COUNTRY: France
REGION: Europe
POPULATION: 2148000
BEST_TIME_TO_VISIT: April to June and September to October
LANGUAGE: French
CURRENCY: Euro (EUR)
TOP_ATTRACTIONS: Eiffel Tower, Louvre Museum, Notre-Dame Cathedral
POPULAR_FOODS: Croissants, Baguette, Escargots
TRANSPORT: Metro, buses, trams
AVERAGE_DAILY_COST: 150 USD
SAFETY_LEVEL: Medium to High
NOTES: Global center for art and culture.

